# Book Recommendation System

Three recommendation approaches implemented from scratch on real-world datasets:

1. **Popularity-based** — IMDB-style weighted rating, no personalization
2. **Collaborative filtering** — item-based, using cosine similarity on user rating patterns
3. **Content-based filtering** — genre matching using a sparse genre matrix

See `README.md` for dataset download instructions before running this notebook.

In [1]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict
from scipy import sparse
from IPython.display import Image, display

DATA_DIR = "../data"
MODELS_DIR = "../models"

## 1. Load data

Expects `Books.csv`, `Ratings.csv`, `Users.csv` (Book-Crossing dataset) in `data/` — see README for download links.

In [2]:
books = pd.read_csv(f"{DATA_DIR}/Books.csv")
ratings = pd.read_csv(f"{DATA_DIR}/Ratings.csv")
users = pd.read_csv(f"{DATA_DIR}/Users.csv")

books.dropna(inplace=True)
print(f"{len(books):,} books, {len(ratings):,} ratings, {len(users):,} users")

/tmp/ipykernel_545/622632967.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(f"{DATA_DIR}/Books.csv")


271,353 books, 1,149,780 ratings, 278,858 users


## 2. Popularity-based model

Uses the IMDB weighted rating formula so books with very few ratings don't
dominate just because their small sample happened to average high:

**WR = (v ÷ (v+m)) × R + (m ÷ (v+m)) × C**

- R = average rating for the book
- v = number of ratings the book has
- m = minimum ratings required to be considered (99th percentile here)
- C = mean rating across the whole dataset

In [3]:
# Keep only valid 10-digit ISBNs and drop unrated (0) entries
books = books[books["ISBN"].apply(lambda x: len(x) == 10)]
unique_isbn = set(books["ISBN"].unique())
ratings = ratings[ratings["ISBN"].isin(unique_isbn)]
ratings = ratings[ratings["bookRating"] != 0]

In [4]:
avg_rating = (
    ratings.groupby("ISBN")["bookRating"]
    .agg(avg_rating="mean", num_ratings="count")
    .reset_index()
)

m = avg_rating["num_ratings"].quantile(0.99)
C = ratings["bookRating"].mean()

def weighted_rating(row, m=m, C=C):
    v = row["num_ratings"]
    R = row["avg_rating"]
    return (v / (v + m) * R) + (m / (m + v) * C)

avg_rating["weighted_rating"] = avg_rating.apply(weighted_rating, axis=1)
avg_rating.sort_values("weighted_rating", ascending=False, inplace=True)
avg_rating.to_csv(f"{MODELS_DIR}/avg_rating.csv", index=False)
avg_rating.head(10)

,ISBN,avg_rating,num_ratings,weighted_rating
46134,0439139597,9.262774,137,9.018884
24538,0345339738,9.402597,77,8.980597
46424,043935806X,9.033981,206,8.887132
46124,0439136369,9.082707,133,8.860129
68897,059035342X,8.939297,313,8.845817
46123,0439136350,9.035461,141,8.830547
50044,0446310786,8.943925,214,8.811094
24537,0345339711,9.120482,83,8.785423
68896,0590353403,8.983193,119,8.755526
46489,0439425220,9.869565,23,8.724261


### Preview: top 5 books by weighted rating

In [5]:
top_5 = avg_rating.head(5)["ISBN"].values

for isbn in top_5:
    row = books[books["ISBN"] == isbn]
    if row.empty:
        continue
    display(Image(url=row["imageURLM"].values[0]))
    print(row["bookTitle"].values[0])

Harry Potter and the Goblet of Fire (Book 4)


The Return of the King (The Lord of the Rings, Part 3)


Harry Potter and the Order of the Phoenix (Book 5)


Harry Potter and the Prisoner of Azkaban (Book 3)


Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))


## 3. Collaborative filtering (item-based)

Books are compared by *who rated them and how*, not by their content. Two
books are "similar" if the same users tended to rate them similarly.

Steps:
1. Restrict to the 1,000 most-rated books (keeps the user-item matrix a
   manageable size)
2. Pivot into a user × book ratings matrix
3. Standardize each user's ratings (removes the effect of some users rating
   everything higher/lower on average)
4. Compute cosine similarity between books (columns) rather than users

In [6]:
top_books = set(
    avg_rating.sort_values("num_ratings", ascending=False).head(1000)["ISBN"].values
)
filtered_ratings = ratings[ratings["ISBN"].isin(top_books)]

user_item_matrix = filtered_ratings.pivot(
    index="User-ID", columns="ISBN", values="bookRating"
)
user_item_matrix.fillna(0, inplace=True)

def standardize(row):
    denom = row.max() - row.min()
    return (row - row.mean()) / denom if denom != 0 else row * 0

user_item_matrix = user_item_matrix.apply(standardize)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns,
)

with open(f"{MODELS_DIR}/item_similarity_df.pkl", "wb") as f:
    pickle.dump(item_similarity_df, f)

item_similarity_df.head()

ISBN,000649840X,002542730X,0028604199,006000438X,0060008032,0060096195,006016848X,0060173289,0060175400,0060175966,...,1592400876,1844262553,1857022424,1878424319,1931561648,3257229534,3404148665,3423202327,3442541751,3492045170
ISBN,,,,,,,,,,,,,,,,,,,,,
000649840X,1.000000,0.057868,-0.001380,0.066046,0.027201,-0.001766,0.042187,-0.001828,0.018715,-0.001410,...,0.020247,0.018746,0.055157,0.013702,0.028201,-0.001350,0.030416,0.025536,0.024752,0.029911
002542730X,0.057868,1.000000,0.244047,0.009532,0.013645,-0.002424,0.011793,-0.002510,0.022059,-0.001935,...,0.023771,0.008792,0.012553,-0.002526,0.014157,-0.001853,0.015420,0.012732,0.012312,0.015079
0028604199,-0.001380,0.244047,1.000000,-0.001261,-0.001277,0.021499,-0.001619,-0.001648,0.016111,-0.001271,...,0.022598,-0.001546,-0.001282,-0.001659,-0.001313,-0.001217,-0.001252,-0.001284,-0.001277,-0.001323
006000438X,0.066046,0.009532,-0.001261,1.000000,0.023070,-0.001614,0.037972,-0.001671,0.015801,0.056601,...,0.017124,0.015853,0.021369,0.013442,0.023919,-0.001234,0.025809,0.021652,0.020986,0.025374
0060008032,0.027201,0.013645,-0.001277,0.023070,1.000000,0.034914,0.062377,-0.001693,0.022187,-0.001305,...,0.046627,0.022127,0.048832,-0.001703,0.033026,-0.001250,0.035578,0.029929,0.029020,0.035011


In [8]:
def get_recommendations(book_ratings, top_n=10):
    """Given {isbn: user_rating}, return the top_n most similar unrated books."""
    scored = []
    for isbn, rating in book_ratings.items():
        if isbn not in item_similarity_df.columns:
            continue
        similar = item_similarity_df[isbn] * (rating - 5)
        scored.append(similar)

    if not scored:
        return []

    combined = pd.concat(scored, axis=1).sum(axis=1)
    combined = combined.sort_values(ascending=False)

    recommendations = []
    for isbn in combined.index:
        if isbn not in book_ratings:
            recommendations.append(isbn)
        if len(recommendations) == top_n:
            break
    return recommendations

### Example: recommend books based on a small rating history

(Sample ratings below — swap in your own ISBNs and scores to try it out.)

In [9]:
example_ratings = {
    "059035342X": 9,   # Harry Potter and the Sorcerer's Stone
    "0345370775": 10,
    "044021145X": 8,
    "0440214041": 10,
    "0440211727": 7,
}

recommended = get_recommendations(example_ratings)
for isbn in recommended:
    row = books[books["ISBN"] == isbn]
    if row.empty:
        continue
    display(Image(url=row["imageURLM"].values[0]))
    print(row["bookTitle"].values[0])

The Client


Harry Potter and the Chamber of Secrets (Book 2)


Silence of the Lambs


The Chamber


The Rainmaker


Postmortem


Harry Potter and the Prisoner of Azkaban (Book 3)


The Runaway Jury


Harry Potter and the Goblet of Fire (Book 4)


The Lost World


## 4. Content-based filtering (genre matching)

Independent of user rating history — recommends books purely by genre
overlap with what the user says they like.

A `(books × genres)` matrix stores each book's rating in every genre it
belongs to. Since a book only has a handful of genres out of ~980 possible
ones, this matrix is **99%+ zeros** — stored as a `scipy.sparse` matrix
instead of a dense NumPy array (previously 394MB, now under 1MB).

In [10]:
bbe = pd.read_csv(f"{DATA_DIR}/books_1.Best_Books_Ever.csv")
genre_data = bbe[["isbn", "genres", "rating"]].copy()
genre_data["genres"] = genre_data["genres"].apply(eval)

all_genres = sorted({genre for genres in genre_data["genres"] for genre in genres})
genre_index = {genre: i for i, genre in enumerate(all_genres)}

with open(f"{MODELS_DIR}/genres_order.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(all_genres))

print(f"{len(all_genres)} unique genres across {len(genre_data)} books")

982 unique genres across 52478 books


In [11]:
rows, cols, vals = [], [], []
for i, (genres, rating) in enumerate(zip(genre_data["genres"], genre_data["rating"])):
    for genre in genres:
        rows.append(i)
        cols.append(genre_index[genre])
        vals.append(rating)

genre_matrix = sparse.csr_matrix(
    (vals, (rows, cols)), shape=(len(genre_data), len(all_genres))
)

sparse.save_npz(f"{MODELS_DIR}/genre_matrix_sparse.npz", genre_matrix)
print("Matrix shape:", genre_matrix.shape, "| non-zero entries:", genre_matrix.nnz)

Matrix shape: (52478, 982) | non-zero entries: 407718


In [12]:
def recommend_by_genre(liked_genres, top_n=5):
    """Given a list of genre names the user likes, return the top_n best-matching books."""
    preference_vector = np.zeros(len(all_genres))
    for genre in liked_genres:
        if genre in genre_index:
            preference_vector[genre_index[genre]] = 1

    scores = genre_matrix.dot(preference_vector)
    top_indices = np.argsort(scores)[::-1][:top_n]
    return bbe.iloc[top_indices][["title", "author", "rating", "coverImg"]]

### Example: recommend books by genre preference

In [13]:
recommend_by_genre(["Fiction", "Romance", "Magic", "Vampires", "Action"])

,title,author,rating,coverImg
49685,Act of Passion,Mandy M. Roth (Goodreads Author),4.63,https://i.gr-assets.com/images/S/compressed.ph...
22946,A Court of Thorns and Roses eSampler,Sarah J. Maas (Goodreads Author),4.60,https://i.gr-assets.com/images/S/compressed.ph...
46338,A Ride of Peril,Bella Forrest (Goodreads Author),4.60,https://i.gr-assets.com/images/S/compressed.ph...
46524,A Meet of Tribes,Bella Forrest (Goodreads Author),4.59,https://i.gr-assets.com/images/S/compressed.ph...
22152,"The Morganville Vampires, #1-9",Rachel Caine (Goodreads Author),4.58,https://i.gr-assets.com/images/S/compressed.ph...
